단계별 파이프라인
1. PDF 페이지별 추출
2. PDF 페이지별 텍스트를 토큰 단위로 분할해서 청크 생성
3. 임베딩
4. Qdrant 컬렉션 생성
5. Qdrant 인덱싱
6. Qdrant 검색결과 JSON 변환

하이라이팅 추가 예정
- 검색 결과에서 핵심 구절을 강조 표시
- 출처 링크와 함께 표시해 신뢰성 확보

요약 추가 예정
- 강조된 구절을 기반으로 간결한 요약 생성
- 중복 제거 및 핵심 메시지 강화

번역 추가 예정
- 요약된 내용을 원하는 언어로 변환
- 다국어 지원 및 글로벌 활용성 확보

Q&A 추가 예정
- 사용자의 질문에 맞춰 응답 생성
- 검색된 문서와 요약을 기반으로 정확한 답변 제공

LLM 추가 예정
- 최종적으로 대형 언어 모델이 맥락을 반영해 답변 강화
- 자연스러운 문장 흐름과 풍부한 설명으로 마무리

1단계: PDF 텍스트 추출
- 라이브러리 설치: !pip install pdfplumber

In [15]:
# PDF 텍스트 추출 라이브러리 import
import pdfplumber

# PDF 파일 열기: 튜플 형태 키:값 예시) page: 1, text: pdf 원문
pdf_path = "sample_korean_long.pdf"
page_texts = []
with pdfplumber.open(path_or_fp=pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        page_texts.append({
            "page": i+1, 
            "text": text,
            "pdf_link": f"{pdf_path}#page={i+1}"
        })

# 텍스트 추출 확인
print("총 텍스트 길이:", len(page_texts))
print(page_texts[:500]) # 앞부분만 확인

총 텍스트 길이: 4
[{'page': 1, 'text': '제목: 인공지능 개요\n인공지능은 현대 사회에서 다양한 분야에 적용되고 있습니다. 의료, 금융, 제조업, 교육 등 거의 모든 산업에서\n인공지능 기술이 활용되고 있으며, 특히 대규모 언어 모델(LLM)은 방대한 텍스트 데이터를 학습하여 사람과\n유사한 방식으로 질문에 답변할 수 있습니다. 이러한 모델은 단순한 질의응답을 넘어 문서 요약, 번역, 창의적\n글쓰기, 코드 생성 등 다양한 작업을 수행할 수 있어 실무에서 큰 가치를 제공합니다.\n인공지능의 발전은 1950년대 앨런 튜링의 튜링 테스트에서 시작되었습니다. 이후 전문가 시스템, 신경망,\n머신러닝, 딥러닝으로 이어지는 발전 과정은 방대한 데이터를 처리하고 학습하는 능력을 강화했습니다. 오늘날\n인공지능은 의료 진단, 금융 거래 분석, 자율주행, 자연어 처리, 이미지 인식 등 다양한 분야에서 활용되고\n있습니다.\n또한 인공지능은 윤리적 문제와 사회적 영향도 동반합니다. 데이터 프라이버시, 알고리즘 편향, 자동화로 인한\n일자리 변화 등은 사회적 논의가 필요한 영역입니다. 인공지능의 미래는 기술적 발전뿐 아니라 사회적 합의와\n규제 체계에 의해 결정될 것입니다.\n(여기에 추가 설명을 반복적으로 확장하여 1000 토큰 이상 분량으로 늘려주세요. 예: 인공지능의 역사, 주요\n기술, 사회적 영향, 미래 전망 등)', 'pdf_link': 'sample_korean_long.pdf#page=1'}, {'page': 2, 'text': '제목: 벡터 데이터베이스\n벡터 데이터베이스는 인공지능과 함께 사용될 때 강력한 도구가 됩니다. 문서, 이미지, 오디오 데이터를 벡터로\n변환하여 저장하면 질문과 가장 유사한 데이터를 빠르게 검색할 수 있습니다. 이는 단순 키워드 검색보다 훨씬\n정교한 결과를 제공합니다.\n벡터 검색은 코사인 유사도, 유클리드 거리, 내적(dot product) 등을 활용하여 벡터 간의 유사성을 계산합니다.\n예를 들어 "고양이 사진

2단계: PDF 페이지별 텍스트를 토큰 단위로 분할해서 청크 생성
- 문단 단위 대신 토큰/문자 기준으로 일정 길이로 나누어 문맥 손실을 최소화
- overlap을 주어 청크 사이 문맥을 이어준다

In [16]:
# transformers AutoTokenizer 라이브러리 import
from transformers import AutoTokenizer

# AutoTokenizer 모델: 토큰 개수는 모델의 토큰 사전(vocabulary)과 토큰화 규칙에 따라 달라짐
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 토큰 함수 생성
def chunk_text_by_tokens(page_text, max_tokens=256, overlap=50):
    # 텍스트를 토큰 ID 리스트로 변환
    tokens = tokenizer.encode(page_text, add_special_tokens=False)
    chunks = []
    step = max_tokens - overlap
    # 두번째 루프 전에 206(256-50) step으로 증가 적용, 0번째 0:256, 1번째 206:462
    for i in range(0, len(tokens), step):
        chunk = tokens[i:i + max_tokens] # 토큰 범위 지정
        chunks.append(tokenizer.decode(chunk)) # 토큰을 다시 문자열로 변환
    return chunks

# 전체 PDF 페이지 처리
all_chunks = []
for page in page_texts:
    page_number = page["page"] # 페이지 번호 추출
    page_text = page["text"] # 페이지 텍스트 추출

    # 페이지 텍스트를 토큰 단위로 청크 분할
    page_chunks = chunk_text_by_tokens(page_text, max_tokens=256, overlap=50)

    # 페이지 번호를 메타데이터로 함께 저장
    for j, chunk in enumerate(page_chunks):
        all_chunks.append({
            "page": page_number,
            "chunk_index": j, # 페이지 내 청크 번호
            "text": chunk,
            "pdf_link": f"{pdf_path}#page={page_number}" # 페이지 출처 정보
        })

print(len(all_chunks), "개의 청크 생성")
print(all_chunks)

8 개의 청크 생성
[{'page': 1, 'chunk_index': 0, 'text': '제목: 인공지능 개요 인공지능은 현대 사회에서 다양한 분야에 적용되고 있습니다. 의료, 금융, 제조업, 교육 등 거의 모든 산업에서 인공지능 기술이 활용되고 있으며, 특히 대규모 언어 모델(LLM)은 방대한 텍스트 데이터를 학습하여 사람과 유사한 방식으로 질문에 답변할 수 있습니다. 이러한 모델은 단순한 질의응답을 넘어 문서 요약, 번역, 창의적 글쓰기, 코드 생성 등 다양한 작업을 수행할 수 있어 실무에서 큰 가치를 제공합니다. 인공지능의 발전은 1950년대 앨런 튜링의 튜링 테스트에서 시작되었습니다. 이후 전문가 시스템, 신경망, 머신러닝, 딥러닝으로 이어지는 발전 과정은 방대한 데이터를 처리하고 학습하는 능력을 강화했습니다. 오늘날 인공지능은 의료 진단, 금융 거래 분석, 자율주행, 자연어 처리, 이미지 인식 등 다양한 분야에서 활용되고 있습니다. 또한 인공지능은 윤리적 문제와 사회적 영향도 동반합니다. 데이터 프라이버시, 알고리즘 편향, 자동화로 인한 일자리 변화 등은 사회적 논의가 필요한 영역입니다. 인공지능의 미래는 기술적 발전뿐 아니라 사회적 합의와 규제 체계에 의해 결정될 것입니다. (여기에 추가 설명을', 'pdf_link': 'sample_korean_long.pdf#page=1'}, {'page': 1, 'chunk_index': 1, 'text': '시, 알고리즘 편향, 자동화로 인한 일자리 변화 등은 사회적 논의가 필요한 영역입니다. 인공지능의 미래는 기술적 발전뿐 아니라 사회적 합의와 규제 체계에 의해 결정될 것입니다. (여기에 추가 설명을 반복적으로 확장하여 1000 토큰 이상 분량으로 늘려주세요. 예: 인공지능의 역사, 주요 기술, 사회적 영향, 미래 전망 등)', 'pdf_link': 'sample_korean_long.pdf#page=1'}, {'page': 2, 'chunk_index': 0, 'text': '제목: 벡터 데이터베이스 벡

3단계: 임베딩 단계
- SentenceTransformer 같은 모델을 사용해 각 청크를 고차원 벡터(embedding)로 변환
- 이렇게 얻은 벡터는 의미적 유사성을 계산할 수 있어서, 검색(RAG), 분류, 추천 등에 활용

In [17]:
# SentenceTransformer 임베딩 라이브러리 import
from sentence_transformers import SentenceTransformer

# 다국어 지원 SentenceTransformer 모델 로드
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

embeddings = []
for i, chunk in enumerate(all_chunks):
    # embedding 후에 리스트 형태로 담는다
    embedding = model.encode(chunk["text"]).tolist()
    embeddings.append(embedding)
    
# 결과 확인
print("임베딩 벡터 개수:", len(embeddings)) # 청크 개수와 동일
print("첫번째 청크 임베딩 벡터 크기:", len(embeddings[0])) # 일반적으로 384 차원

임베딩 벡터 개수: 8
첫번째 청크 임베딩 벡터 크기: 384


4단계: Qdrant 컬렉션 생성
- 라이브러리 설치: !pip install qdrant-client

In [18]:
from qdrant_client import QdrantClient # Qdrant 클라이언트 import
from qdrant_client.http import models # Qdrant 클라이언트에서 서버와 통신할때 타입 정의 벡터 파라미터, PointStruct 구조체(id/vector/payload) 설정 모델 불러오는 역할 import

# Qdrant 서버 연결
client = QdrantClient(host="localhost", port=6333)

collection_name = "documents_pdf"

# 컬렉션 존재 여부 확인
try:
    client.get_collection(collection_name=collection_name)
    print(f"{collection_name} 컬렉션이 이미 존재합니다.")
except Exception:
    # 컬렉션이 없으면 새로 생성
    client.create_collection(
        collection_name=collection_name,
        # models.VectorParams(size 벡터 크기, distance 거리 계산 방식 정의)
        vectors_config=models.VectorParams(size=len(embeddings[0]), distance=models.Distance.COSINE),
    )
    print(f"{collection_name} 컬렉션이 새로 생성되었습니다.")

documents_pdf 컬렉션이 이미 존재합니다.


5단계: Qdrant 인덱싱
- Upsert 인덱싱 작업

In [19]:
# points 리스트 생성
points = []
for i, chunk in enumerate(all_chunks):
    points.append(
        # Qdrant에 저장할 데이터 구조체
        models.PointStruct(
            id=i, # 각 벡터의 고유 식별자(여기서는 인덱스 번호 i)
            vector=embeddings[i], # 실제 임베딩 벡터
            payload={
                "page": chunk["page"], # 페이지 정보
                "chunk_index": chunk["chunk_index"], # 각 페이지내에서 분할된 인덱스 번호
                "text": chunk["text"], # 텍스트 원문
                "pdf_link": chunk["pdf_link"] # pdf 출처 링크
            } # 벡터와 함께 저장할 메타데이터(여기서는 원문 청크 텍스트)
        )
    )

# upsert(insert+update): 새 데이터를 추가하거나 기존 데이터를 덮어씀
client.upsert(
    collection_name=collection_name, 
    points=points
) # 컬렉션에 저장, PointStruct 리스트를 전달->Qdrant DB 벡터와 텍스트 저장

print("인덱싱 완료")

인덱싱 완료


6단계: Qdrant 검색
- 검색 결과: JSON 변환

In [22]:
import json

# 질의문
query = "벡터 데이터베이스란 무엇인가?"

# 질의문 벡터 변환: SentenceTransformers 모델을 사용하여 질의문을 임베딩 벡터로 변환, 넘파이 배열을 파이썬 리스트로 반환(Qdrant 클라이언트는 리스트 타입을 요구함)
query_vector = model.encode(sentences=query).tolist() # 질의문 -> 벡터 변환(타입은 리스트)

# Qdrant 검색: query_points 호출시 Qdrant 서버가 검색 결과 개체를 반환
results = client.query_points( # query_points 메서드로 Qdrant DB에서 검색 실행
    collection_name=collection_name, # 컬렉션명
    query=query_vector, # 질의문 벡터
    limit=3 # 가장 유사한 상위 3건 결과 반환
)

# 검색 결과를 JSON 변환
output = {
    "query": query,
    "results": []
}

for r in results.points:
    output["results"].append({
        "page": r.payload["page"],
        "chunk_index": r.payload["chunk_index"],
        "text": r.payload["text"],
        "score": r.score,
        "pdf_link": r.payload["pdf_link"]
    })

# JSON 문자열로 변환 후 출력
# json.dumps(): dict, list 등을 JSON 문자열로 변환
# ensure_ascii=False: 아스키코드 변환 안되게 해서 한글 출력
# indent=2: 2칸 들여쓰기
print(json.dumps(output, ensure_ascii=False, indent=2))

# 결과 확인: 검색결과 객체 안에는 여러 속성이 있으며, 그중 핵심이 바로 points 이다
# for r in results.points: # 검새된 포인트(PointStruct)들의 리스트이다
#     # 검색결과 원문(페이지/내용), 질의문 벡터와 해당 청크 벡터의 유사도 점수 출력
#     print("Page", r.payload["page"],
#           "Chunk", r.payload["chunk_index"],
#           "=>", r.payload["text"][:120], "...", # 텍스트 일부만 표시
#           "score", r.score)
#     print("출처 링크:", r.payload["pdf_link"])
#     print("-" * 80)

{
  "query": "벡터 데이터베이스란 무엇인가?",
  "results": [
    {
      "page": 2,
      "chunk_index": 0,
      "text": "제목: 벡터 데이터베이스 벡터 데이터베이스는 인공지능과 함께 사용될 때 강력한 도구가 됩니다. 문서, 이미지, 오디오 데이터를 벡터로 변환하여 저장하면 질문과 가장 유사한 데이터를 빠르게 검색할 수 있습니다. 이는 단순 키워드 검색보다 훨씬 정교한 결과를 제공합니다. 벡터 검색은 코사인 유사도, 유클리드 거리, 내적(dot product) 등을 활용하여 벡터 간의 유사성을 계산합니다. 예를 들어 \"고양이 사진\"을 검색하면 키워드 기반 검색은 단순히 ‘고양이’라는 단어가 포함된 문서를 반환하지만, 벡터 검색은 실제로 고양이 이미지와 유사한 벡터를 찾아냅니다. Qdrant, Pinecone, Weaviate 등 다양한 벡터 데이터베이스가 존재하며, 특히 Qdrant는 고성능 검색과 인덱싱을 지원하여 RAG 시스템 구축에 널리 활용됩니다. 벡터 데이터베이스는 추천 시스템, 이미지 검색, 음성 인식, 법률 문서 검색 등 다양한 분야에서 활용됩니다. (여기에 벡터화 과정, 검색 방식, 활용 사례, 키워드 검색과의 비교 등을 반복적으로 확장하여 1000 토큰 이상 분",
      "score": 0.75341845,
      "pdf_link": "sample_korean_long.pdf#page=2"
    },
    {
      "page": 2,
      "chunk_index": 1,
      "text": "베이스는 추천 시스템, 이미지 검색, 음성 인식, 법률 문서 검색 등 다양한 분야에서 활용됩니다. (여기에 벡터화 과정, 검색 방식, 활용 사례, 키워드 검색과의 비교 등을 반복적으로 확장하여 1000 토큰 이상 분량으로 늘려주세요.)",
      "score": 0.3716484,
      "pdf_link": "sample_korean_long